# 31.02 Глобальная чувствительность двуслойной модели

> **Статус:** канонический синтетический сценарный расчёт индексов Соболя
> первого и полного порядков. Экспериментальные данные не используются;
> результат не является оценкой тканей добровольцев, погрешности прибора или
> идентифицируемости обратной задачи.

Вход — проверенное ядро `two_layer_model.py`/`30.04`. Определения и ограничения
даны в `31.00`; локальные производные рассматриваются отдельно в `31.01`.


## Происхождение и исправленная граница вывода

Исторический источник — Sobol-раздел
`archive/legacy/31.90_Объединённый_анализ_чувствительности.ipynb` (ранее файл
`05_Анализ_чувствительности_решётки.ipynb`). В нём использовались независимые
равномерные диапазоны $\rho_1=3\ldots7$ Ом·м, $\rho_2=15\ldots25$ Ом·м и
$h=5\ldots45$ мм, выборка Saltelli с базовым размером 256 и только индексы
первого порядка. Происхождение этих диапазонов, сходимость и неопределённость
индексов не были проверены, а сохранённых численных outputs нет. Поэтому старое
утверждение, что расчёт «подтверждает» преимущество крупных сборок и главную
роль $h$, не переносится как результат.

Ниже эти интервалы сохранены только как **унаследованный широкий учебный
сценарий**, а не как физиологические нормативы. Второй сценарий отличается
только более узким интервалом $h=15\ldots25$ мм и нужен, чтобы прямо показать
зависимость ранжирования от принятого распределения. Ни один сценарий пока не
калиброван по КТ или электрическим данным.


## Математическая постановка

Для фиксированного размера $L=2a$ и формы $\beta=b/a=0{,}5$ рассматривается
подписанный аналитический импеданс

$$Y=Z(\rho_1,\rho_2,h;L,\beta).$$

В каждом сценарии три входа считаются **независимыми** и равномерно
распределёнными на явно заданных интервалах. Это вычислительное допущение,
не подтверждённое свойство физиологических параметров. При наличии корреляций
классические индексы Соболя теряют эту интерпретацию и должны быть заменены
методом для зависимых входов.

$$
S_j=\frac{\operatorname{Var}_{\theta_j}
\left(\mathbb E[Y\mid\theta_j]\right)}{\operatorname{Var}(Y)},
\qquad
S_{Tj}=1-\frac{\operatorname{Var}_{\theta_{\sim j}}
\left(\mathbb E[Y\mid\theta_{\sim j}]\right)}{\operatorname{Var}(Y)}.
$$

Интегралы вычисляются детерминированной тензорной квадратурой Гаусса—Лежандра.
Это убирает случайный Monte Carlo-разброс и зависимость от `SALib`, но не
устраняет неопределённость самих диапазонов и допущения независимости.
Сходимость среднего, дисперсии и индексов контролируется последовательными
порядками квадратуры 8, 10 и 12. Допуски этого контроля — только критерии
численной воспроизводимости данного кода, а не заявленная точность физической
модели или эксперимента.

Индексы описывают доли **модельной дисперсии $Z$ при заданных распределениях**.
Они не являются долями тока по тканям, экспериментальным «шумом», причинной
важностью параметров, точностью оценки $\rho_2$ или критерием выбора пары
сборок. Последовательность реальных записей и ошибки установки здесь не
моделируются.


In [ ]:
from pathlib import Path
import platform
import sys

import numpy as np

candidates = [Path.cwd(), Path.cwd() / "Colab Notebooks", Path.cwd().parent]
model_paths = sorted({
    (candidate / "two_layer_model.py").resolve()
    for candidate in candidates
    if (candidate / "two_layer_model.py").is_file()
})
if len(model_paths) != 1:
    raise RuntimeError(f"expected one canonical two_layer_model.py, found: {model_paths}")

model_path = model_paths[0]
sys.path.insert(0, str(model_path.parent))
import two_layer_model as tlm

if Path(tlm.__file__).resolve() != model_path:
    raise RuntimeError(f"imported non-canonical model: {tlm.__file__}")

geometry_from_size = tlm.geometry_from_size
transfer_impedance = tlm.transfer_impedance

print(f"Python {platform.python_version()}; NumPy {np.__version__}")
print(f"Модель: {model_path}")


In [ ]:
PARAMETERS = ("rho1", "rho2", "h")
UNITS = {"rho1": "Ом·м", "rho2": "Ом·м", "h": "м"}
BETA = 0.5
SIZES_M = np.arange(0.050, 0.141, 0.010)  # реально изготовленный ряд 50...140 мм
MODEL_VALIDITY = "unverified_without_subject_specific_CT_FEM_Lmax"

SCENARIOS = {
    "унаследованный широкий": {
        "status": "illustrative_unvalidated",
        "source": "archive/legacy/31.90 §7; происхождение диапазонов не подтверждено",
        "bounds": {
            "rho1": (3.0, 7.0),
            "rho2": (15.0, 25.0),
            "h": (0.005, 0.045),
        },
    },
    "иллюстрация узкого h": {
        "status": "illustrative_unvalidated",
        "source": "синтетическая проверка зависимости индексов от ширины h",
        "bounds": {
            "rho1": (3.0, 7.0),
            "rho2": (15.0, 25.0),
            "h": (0.015, 0.025),
        },
    },
}

print(f"model_validity={MODEL_VALIDITY}")
for scenario_name, scenario in SCENARIOS.items():
    print(f"{scenario_name} [{scenario['status']}]")
    print(f"  источник: {scenario['source']}")
    for name in PARAMETERS:
        lo, hi = scenario["bounds"][name]
        scale = 1000.0 if name == "h" else 1.0
        unit = "мм" if name == "h" else UNITS[name]
        print(f"  {name}: {lo * scale:g}...{hi * scale:g} {unit}; U(a,b), независимо")


In [ ]:
def _integrate_all(values, weights):
    result = np.asarray(values, dtype=float)
    for axis in range(result.ndim - 1, -1, -1):
        result = np.tensordot(result, weights[axis], axes=(axis, 0))
    return float(result)


def _conditional_expectation(values, weights, keep_axes):
    result = np.asarray(values, dtype=float)
    remove_axes = sorted(set(range(result.ndim)) - set(keep_axes), reverse=True)
    for axis in remove_axes:
        result = np.tensordot(result, weights[axis], axes=(axis, 0))
    return result


def _uniform_quadrature(lower, upper, order):
    if not np.isfinite([lower, upper]).all() or not lower < upper:
        raise ValueError("uniform bounds must be finite and strictly increasing")
    nodes, weights = np.polynomial.legendre.leggauss(order)
    mapped = (lower + upper) / 2.0 + (upper - lower) * nodes / 2.0
    return mapped, weights / 2.0


def _sobol_from_tensor(values, weights):
    mean = _integrate_all(values, weights)
    variance = _integrate_all((values - mean) ** 2, weights)
    if not np.isfinite(variance) or variance <= 0:
        raise RuntimeError("non-positive output variance: Sobol indices are undefined")

    first = []
    total = []
    for axis in range(values.ndim):
        conditional_one = _conditional_expectation(values, weights, [axis])
        first.append(float(np.dot(weights[axis], (conditional_one - mean) ** 2) / variance))

        complement = [j for j in range(values.ndim) if j != axis]
        conditional_rest = _conditional_expectation(values, weights, complement)
        complement_weights = [weights[j] for j in complement]
        variance_rest = _integrate_all((conditional_rest - mean) ** 2, complement_weights)
        total.append(float(1.0 - variance_rest / variance))
    return mean, variance, np.asarray(first), np.asarray(total)


def sobol_by_quadrature(size_m, bounds, order=12):
    nodes = []
    weights = []
    for name in PARAMETERS:
        lo, hi = bounds[name]
        parameter_nodes, parameter_weights = _uniform_quadrature(lo, hi, order)
        nodes.append(parameter_nodes)
        weights.append(parameter_weights)

    a, b = geometry_from_size(size_m, BETA)
    values = np.empty((order,) * len(PARAMETERS), dtype=float)
    for index in np.ndindex(values.shape):
        rho1, rho2, h = (nodes[axis][i] for axis, i in enumerate(index))
        values[index] = transfer_impedance(rho1, rho2, h, a, b)
    if not np.isfinite(values).all():
        raise RuntimeError("two_layer_model returned a non-finite value")

    mean, variance, first, total = _sobol_from_tensor(values, weights)
    return {
        "mean": mean,
        "variance": variance,
        "S1": dict(zip(PARAMETERS, first)),
        "ST": dict(zip(PARAMETERS, total)),
    }


def _self_test_quadrature(order=10):
    x, w = _uniform_quadrature(-1.0, 1.0, order)
    transformed, transformed_weights = _uniform_quadrature(2.0, 5.0, order)
    np.testing.assert_allclose(np.sum(transformed_weights), 1.0, rtol=0.0, atol=1e-14)
    np.testing.assert_allclose(np.dot(transformed_weights, transformed), 3.5, rtol=0.0, atol=1e-14)
    assert np.all((transformed > 2.0) & (transformed < 5.0))

    xx, yy, zz = np.meshgrid(x, x, x, indexing="ij")
    weights = [w, w, w]

    # Аддитивный пример: S1 == ST == доле дисперсии каждого слагаемого.
    additive = xx + 2.0 * yy + 3.0 * zz
    _, _, first, total = _sobol_from_tensor(additive, weights)
    expected = np.array([1.0, 4.0, 9.0]) / 14.0
    np.testing.assert_allclose(first, expected, rtol=1e-12, atol=1e-12)
    np.testing.assert_allclose(total, expected, rtol=1e-12, atol=1e-12)

    # Чистое взаимодействие x*y: S1=(0,0,0), ST=(1,1,0).
    interaction = xx * yy + 0.0 * zz
    _, _, first, total = _sobol_from_tensor(interaction, weights)
    np.testing.assert_allclose(first, [0.0, 0.0, 0.0], rtol=0.0, atol=1e-12)
    np.testing.assert_allclose(total, [1.0, 1.0, 0.0], rtol=0.0, atol=1e-12)


_self_test_quadrature()
print("Самотест квадратуры: пройден")


In [ ]:
QUADRATURE_ORDERS = (8, 10, 12)
INDEX_ATOL = 1e-4
MOMENT_RTOL = 1e-4
BOUND_ATOL = 1e-10

results = {}
convergence = {}

for scenario_name, scenario in SCENARIOS.items():
    results[scenario_name] = {}
    worst_index_difference = 0.0
    worst_moment_relative_difference = 0.0
    for size_m in SIZES_M:
        estimates = {
            order: sobol_by_quadrature(size_m, scenario["bounds"], order=order)
            for order in QUADRATURE_ORDERS
        }
        fine = estimates[QUADRATURE_ORDERS[-1]]
        results[scenario_name][float(size_m)] = fine

        for lower_order, upper_order in zip(QUADRATURE_ORDERS[:-1], QUADRATURE_ORDERS[1:]):
            lower = estimates[lower_order]
            upper = estimates[upper_order]
            for moment in ("mean", "variance"):
                relative = abs(upper[moment] - lower[moment]) / max(abs(upper[moment]), 1e-15)
                worst_moment_relative_difference = max(worst_moment_relative_difference, relative)
            for family in ("S1", "ST"):
                for name in PARAMETERS:
                    difference = abs(upper[family][name] - lower[family][name])
                    worst_index_difference = max(worst_index_difference, difference)

        first = np.array([fine["S1"][name] for name in PARAMETERS])
        total = np.array([fine["ST"][name] for name in PARAMETERS])
        if not np.isfinite(np.r_[fine["mean"], fine["variance"], first, total]).all():
            raise RuntimeError("non-finite moment or Sobol index")
        if np.any(first < -BOUND_ATOL) or np.any(total > 1.0 + BOUND_ATOL):
            raise RuntimeError("Sobol index outside [0, 1] beyond numerical tolerance")
        if np.any(first > total + BOUND_ATOL) or first.sum() > 1.0 + BOUND_ATOL:
            raise RuntimeError("Sobol decomposition identities failed")

    convergence[scenario_name] = {
        "index_abs": worst_index_difference,
        "moment_rel": worst_moment_relative_difference,
    }

for scenario_name, diagnostics in convergence.items():
    print(
        f"{scenario_name}: max index_diff={diagnostics['index_abs']:.3e}; "
        f"max relative_moment_diff={diagnostics['moment_rel']:.3e}"
    )
    if diagnostics["index_abs"] > INDEX_ATOL or diagnostics["moment_rel"] > MOMENT_RTOL:
        raise RuntimeError("quadrature-order check failed; increase the order")


In [ ]:
for scenario_name, by_size in results.items():
    scenario = SCENARIOS[scenario_name]
    print(f"\n{scenario_name} [{scenario['status']}]")
    print(f"Источник: {scenario['source']}")
    print(f"Применимость плоской модели: {MODEL_VALIDITY}")
    print("L, мм | S1(rho1) ST(rho1) | S1(rho2) ST(rho2) | S1(h) ST(h)")
    for size_m, row in by_size.items():
        chunks = [f"{row['S1'][name]:.4f} {row['ST'][name]:.4f}" for name in PARAMETERS]
        print(f"{size_m * 1000:5.0f} | " + " | ".join(chunks))

print("\nПроверка взаимодействий ST-S1 (минимум по всем случаям):")
minimum_interaction = min(
    row["ST"][name] - row["S1"][name]
    for by_size in results.values()
    for row in by_size.values()
    for name in PARAMETERS
)
print(f"min(ST-S1) = {minimum_interaction:.3e}")
if minimum_interaction < -BOUND_ATOL:
    raise RuntimeError("unexpected negative interaction beyond numerical tolerance")


## Как интерпретировать полученные таблицы

1. Сравнивать можно только строки внутри одного явно заданного сценария.
   Изменение индексов между двумя сценариями показывает зависимость результата
   от ширины принятого интервала $h$, а не изменение физики тканей.
2. $S_j$ — отдельный вклад параметра; $S_{Tj}-S_j$ — его взаимодействия с
   другими входами. Сумма индексов первого порядка может быть меньше единицы.
3. Даже если $S_{\rho_2}$ растёт с $L$ в данном сценарии, это ещё не доказывает
   возможность устойчиво восстановить $\rho_2$ по двум последовательным
   измерениям. Для этого нужны модель наблюдения, калибровка, экспериментальная
   ковариация ошибок и анализ идентифицируемости (`31.03`/`32.*`).
4. Индекс $S_h$ не следует называть «уровнем шума»: здесь $h$ — входной
   параметр прямой модели. Реальные источники погрешности — переклейка
   электродов, вариабельность физиологического состояния, дрейф, gain/offset,
   ошибки геометрии и несоответствие анатомии плоской модели — в расчёт не
   включены.


## Что требуется до научного использования

- заменить учебные интервалы субъектными или популяционными распределениями с
  явным источником: КТ для $h$, независимо обоснованные данные для
  $\rho_1,\rho_2$;
- проверить зависимости между параметрами и при необходимости отказаться от
  классической независимой схемы Соболя;
- задать наблюдаемую прибором величину: подписанный $Z$, модуль или
  калиброванный канал;
- отдельно перенести неопределённость монтажа и последовательной записи в
  модель наблюдения;
- повторить анализ на индивидуальной КТ/FEM-модели и сравнить ранжирование с
  плоскослоистой моделью.

До выполнения этих условий notebook даёт только воспроизводимый условный тест
прямой математической модели и не обосновывает выбор размеров сборок.
